In [36]:
!pip install -q chromadb sentence-transformers langgraph fastapi uvicorn pydantic

In [37]:
import os
import chromadb
from sentence_transformers import SentenceTransformer

os.makedirs("docs", exist_ok=True)

documents = {
    "doc_01.txt": "Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee. Priority delivery, which reserves the next available rider slot, is available at checkout for an additional INR 15. Zepto does not currently deliver to addresses outside its listed serviceable pin codes.",
    "doc_02.txt": "Grocery and perishable items may be reported for a return within 24 hours of delivery if damaged, spoiled, or incorrect; non-perishable packaged items may be returned within 7 days of delivery in unopened, resalable condition. Approved refunds are credited to the original payment method within 3–5 business days, or instantly to the Zepto wallet if the customer opts for wallet credit. Personal care items that have been opened are non-returnable except in the case of a manufacturing defect. Return pickup, where required, is arranged free of cost by Zepto.",
    "doc_03.txt": "Zepto offers three account tiers: Basic (free, default tier, standard delivery fees apply), Zepto Pass (INR 49 per month, free standard delivery on all orders and 5% off select categories), and Zepto Pass+ (INR 99 per month, free priority delivery, 10% off select categories, and early access to limited-time deals 24 hours before they go live to Basic and Pass members). Membership can be cancelled at any time from account settings; cancelling stops the next billing cycle but does not refund the current membership period.",
    "doc_04.txt": "Every Zepto order shows a live rider-tracking map from the moment it is packed until delivery, accessible from the 'Track Order' screen. Estimated delivery time updates automatically as the rider moves. If an order's status shows no movement for more than 20 minutes past its original estimated delivery time, customers should contact support directly rather than continue waiting, since this indicates a likely delivery issue.",
    "doc_05.txt": "Orders can be cancelled free of cost any time before the order status changes to 'Packed', typically within the first 2 minutes of placing the order. Once an order has been packed, it can no longer be cancelled through the app, since the rider is dispatched immediately after packing given Zepto's quick-delivery model. If a packed order cannot be delivered due to a Zepto-side issue (for example, rider unavailability), the order is auto-cancelled and fully refunded without any cancellation fee.",
    "doc_06.txt": "If an order arrives with damaged, spoiled, or missing items, customers must report it within 24 hours of delivery through the 'Report an Issue' button on the order page. Zepto ships a free replacement or issues a full refund for damaged, spoiled, or missing items without requiring the customer to return the original item, unless the order value exceeds INR 1000, in which case a photo of the issue must be submitted through the report form before a replacement or refund is processed.",
    "doc_07.txt": "Zepto gift cards are available in fixed denominations of INR 100, INR 250, INR 500, and INR 1000, and are delivered by email or SMS within minutes of purchase. Gift cards are valid for 1 year from the date of issue and carry no maintenance fees. Gift card balance can be combined with one other payment method at checkout but cannot be combined with another gift card in the same transaction. Gift card balance cannot be redeemed for cash except where required by law.",
    "doc_08.txt": "Zepto customer support is available via in-app chat 24 hours a day, 7 days a week, given the time-sensitive nature of quick commerce deliveries. Average in-app chat response time is under 2 minutes. Email support is also available for non-urgent queries and is answered within 24 hours on business days. Phone support is not offered."
}

for name, text in documents.items():
    with open(os.path.join("docs", name), "w", encoding="utf-8") as file:
        file.write(text)

texts = []
ids = []

for name in sorted(os.listdir("docs")):
    if name.endswith(".txt"):
        with open(os.path.join("docs", name), "r", encoding="utf-8") as file:
            texts.append(file.read().strip())
        ids.append(name.replace(".txt", ""))

model = SentenceTransformer("all-MiniLM-L6-v2")

embeddings = model.encode(
    texts,
    show_progress_bar=True
).tolist()

client = chromadb.EphemeralClient()

collection = client.create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)

collection.add(
    ids=ids,
    documents=texts,
    embeddings=embeddings
)

print("Documents loaded:", len(texts))
print("ChromaDB count:", collection.count())
print("Collection:", collection.name)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Documents loaded: 8
ChromaDB count: 8
Collection: zepto_policies


In [38]:
prompt_template = """
ROLE:
You are a Zepto customer support assistant.

CONTEXT:
Use only the retrieved Zepto policy documents provided as context.

TASK:
Answer the customer question using the relevant retrieved policy information.

FORMAT:
Return a direct and clear answer to the customer.

LENGTH:
Keep the answer concise and preferably between 2 and 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent Zepto policies, prices, delivery rules, refund rules, or support services.

FEW-SHOT EXAMPLE:
Question: What is the delivery fee for orders below INR 149?
Context: Orders below INR 149 incur a flat INR 25 delivery fee.
Answer: Orders below INR 149 have a flat INR 25 delivery fee.

CUSTOMER QUESTION:
{question}

RETRIEVED CONTEXT:
{context}
"""

print(prompt_template)


ROLE:
You are a Zepto customer support assistant.

CONTEXT:
Use only the retrieved Zepto policy documents provided as context.

TASK:
Answer the customer question using the relevant retrieved policy information.

FORMAT:
Return a direct and clear answer to the customer.

LENGTH:
Keep the answer concise and preferably between 2 and 4 sentences.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent Zepto policies, prices, delivery rules, refund rules, or support services.

FEW-SHOT EXAMPLE:
Question: What is the delivery fee for orders below INR 149?
Context: Orders below INR 149 incur a flat INR 25 delivery fee.
Answer: Orders below INR 149 have a flat INR 25 delivery fee.

CUSTOMER QUESTION:
{question}

RETRIEVED CONTEXT:
{context}



In [39]:
import os
from typing import TypedDict
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field

MOCK_LLM = os.getenv("MOCK_LLM", "1")

class AnswerResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0, le=1)

class GraphState(TypedDict):
    query: str
    intent: str
    answer: str
    sources: list[str]
    confidence: float

def classify_intent(state):
    query = state["query"].lower()

    keywords = [
        "delivery",
        "return",
        "refund",
        "membership",
        "tracking",
        "cancel",
        "gift card",
        "support hours"
    ]

    if any(keyword in query for keyword in keywords):
        state["intent"] = "policy_question"
    else:
        state["intent"] = "general_question"

    return state

def retrieve_and_answer(state):
    query = state["query"]

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=3
    )

    retrieved_documents = results["documents"][0]
    retrieved_ids = results["ids"][0]

    top_chunk = retrieved_documents[0]
    top_chunk_snippet = top_chunk[:200]

    if MOCK_LLM == "1":
        answer = f"Based on the retrieved context: {top_chunk_snippet}"
    else:
        context = "\n\n".join(retrieved_documents)
        prompt = prompt_template.format(
            question=query,
            context=context
        )
        answer = top_chunk_snippet

    response = AnswerResponse(
        answer=answer,
        sources=retrieved_ids,
        confidence=1.0
    )

    state["answer"] = response.answer
    state["sources"] = response.sources
    state["confidence"] = response.confidence

    return state

def direct_answer(state):
    response = AnswerResponse(
        answer="I can only answer questions about Zepto policies right now.",
        sources=[],
        confidence=1.0
    )

    state["answer"] = response.answer
    state["sources"] = response.sources
    state["confidence"] = response.confidence

    return state

def route_question(state):
    if state["intent"] == "policy_question":
        return "retrieve_and_answer"
    return "direct_answer"

builder = StateGraph(GraphState)

builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_and_answer", retrieve_and_answer)
builder.add_node("direct_answer", direct_answer)

builder.set_entry_point("classify_intent")

builder.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

graph = builder.compile()

print("Graph created successfully")
print("MOCK_LLM:", MOCK_LLM)
print("ChromaDB count:", collection.count())

Graph created successfully
MOCK_LLM: 1
ChromaDB count: 8


In [40]:
policy_query = "What is the delivery fee for orders below INR 149?"

policy_result = graph.invoke({
    "query": policy_query,
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
})

policy_response = AnswerResponse(
    answer=policy_result["answer"],
    sources=policy_result["sources"],
    confidence=policy_result["confidence"]
)

general_query = "What is the capital of India?"

general_result = graph.invoke({
    "query": general_query,
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
})

general_response = AnswerResponse(
    answer=general_result["answer"],
    sources=general_result["sources"],
    confidence=general_result["confidence"]
)

print("POLICY QUERY")
print(policy_response.model_dump_json(indent=2))

print("\nGENERAL QUERY")
print(general_response.model_dump_json(indent=2))

print("\nPOLICY INTENT:", policy_result["intent"])
print("GENERAL INTENT:", general_result["intent"])

POLICY QUERY
{
  "answer": "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del",
  "sources": [
    "doc_01",
    "doc_05",
    "doc_03"
  ],
  "confidence": 1.0
}

GENERAL QUERY
{
  "answer": "I can only answer questions about Zepto policies right now.",
  "sources": [],
  "confidence": 1.0
}

POLICY INTENT: policy_question
GENERAL INTENT: general_question


In [41]:
policy_query = "What is the delivery fee for orders below INR 149?"

policy_result = graph.invoke({
    "query": policy_query,
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
})

print(policy_result)

general_query = "What is the capital of India?"

general_result = graph.invoke({
    "query": general_query,
    "intent": "",
    "answer": "",
    "sources": [],
    "confidence": 0.0
})

print(general_result)

{'query': 'What is the delivery fee for orders below INR 149?', 'intent': 'policy_question', 'answer': "Based on the retrieved context: Zepto delivers grocery and household essentials to serviceable pin codes within 10 to 30 minutes of order confirmation, depending on the customer's delivery zone and current order volume. Standard del", 'sources': ['doc_01', 'doc_05', 'doc_03'], 'confidence': 1.0}
{'query': 'What is the capital of India?', 'intent': 'general_question', 'answer': 'I can only answer questions about Zepto policies right now.', 'sources': [], 'confidence': 1.0}


In [42]:
main_code = '''
import os
import chromadb
from typing import TypedDict
from sentence_transformers import SentenceTransformer
from langgraph.graph import StateGraph, END
from fastapi import FastAPI
from pydantic import BaseModel, Field

MOCK_LLM = os.getenv("MOCK_LLM", "1")

model = SentenceTransformer("all-MiniLM-L6-v2")

client = chromadb.PersistentClient(path="chroma_db")

collection = client.get_or_create_collection(
    name="zepto_policies",
    metadata={"hnsw:space": "cosine"}
)

prompt_template = """
ROLE:
You are a Zepto customer support assistant.

CONTEXT:
You must use the retrieved Zepto policy documents as the only source of policy information.

TASK:
Answer the customer's question using only the relevant information from the retrieved context.

FORMAT:
Return a direct and clear answer.

LENGTH:
Keep the answer between 2 and 4 sentences when possible.

NEGATIVE CONSTRAINT:
Do not answer using information that is not present in the provided context.
Do not invent Zepto policies, prices, timings, refunds, delivery rules, or support services.

FEW-SHOT EXAMPLE:
Question: What is the delivery fee for orders below INR 149?
Context: Standard delivery is free on orders over INR 149; orders below this threshold incur a flat INR 25 delivery fee.
Answer: Orders below INR 149 have a flat INR 25 standard delivery fee.

CUSTOMER QUESTION:
{question}

RETRIEVED CONTEXT:
{context}
"""

class AskRequest(BaseModel):
    query: str

class AnswerResponse(BaseModel):
    answer: str
    sources: list[str]
    confidence: float = Field(ge=0, le=1)

class GraphState(TypedDict):
    query: str
    intent: str
    answer: str
    sources: list[str]
    confidence: float

def classify_intent(state):
    query = state["query"].lower()

    keywords = [
        "delivery",
        "return",
        "refund",
        "membership",
        "tracking",
        "cancel",
        "gift card",
        "support hours"
    ]

    state["intent"] = "policy_question" if any(keyword in query for keyword in keywords) else "general_question"

    return state

def retrieve_and_answer(state):
    query = state["query"]

    query_embedding = model.encode([query]).tolist()

    results = collection.query(
        query_embeddings=query_embedding,
        n_results=min(3, collection.count())
    )

    documents = results["documents"][0]
    ids = results["ids"][0]

    if not documents:
        return {
            **state,
            "answer": "No relevant Zepto policy information was found.",
            "sources": [],
            "confidence": 0.0
        }

    top_chunk = documents[0]
    snippet = top_chunk[:200]

    if MOCK_LLM == "1":
        answer = f"Based on the retrieved context: {snippet}"
    else:
        context = "\\n\\n".join(documents)
        prompt = prompt_template.format(
            question=query,
            context=context
        )
        answer = snippet

    response = AnswerResponse(
        answer=answer,
        sources=ids,
        confidence=1.0
    )

    return {
        **state,
        "answer": response.answer,
        "sources": response.sources,
        "confidence": response.confidence
    }

def direct_answer(state):
    response = AnswerResponse(
        answer="I can only answer questions about Zepto policies right now.",
        sources=[],
        confidence=1.0
    )

    return {
        **state,
        "answer": response.answer,
        "sources": response.sources,
        "confidence": response.confidence
    }

def route_question(state):
    return "retrieve_and_answer" if state["intent"] == "policy_question" else "direct_answer"

builder = StateGraph(GraphState)

builder.add_node("classify_intent", classify_intent)
builder.add_node("retrieve_and_answer", retrieve_and_answer)
builder.add_node("direct_answer", direct_answer)

builder.set_entry_point("classify_intent")

builder.add_conditional_edges(
    "classify_intent",
    route_question,
    {
        "retrieve_and_answer": "retrieve_and_answer",
        "direct_answer": "direct_answer"
    }
)

builder.add_edge("retrieve_and_answer", END)
builder.add_edge("direct_answer", END)

graph = builder.compile()

app = FastAPI(title="Zepto Support Assistant")

@app.post("/ask", response_model=AnswerResponse)
def ask(request: AskRequest):
    result = graph.invoke({
        "query": request.query,
        "intent": "",
        "answer": "",
        "sources": [],
        "confidence": 0.0
    })

    return AnswerResponse(
        answer=result["answer"],
        sources=result["sources"],
        confidence=result["confidence"]
    )
'''

with open("main.py", "w", encoding="utf-8") as file:
    file.write(main_code)

print("main.py created successfully")

main.py created successfully


In [43]:
dockerfile = """FROM python:3.11-slim

WORKDIR /app

COPY requirements.txt .

RUN pip install --no-cache-dir -r requirements.txt

COPY . .

ENV MOCK_LLM=1

EXPOSE 7860

CMD ["uvicorn", "main:app", "--host", "0.0.0.0", "--port", "7860"]
"""

requirements = """fastapi
uvicorn
langgraph
chromadb
sentence-transformers
pydantic
"""

with open("Dockerfile", "w", encoding="utf-8") as file:
    file.write(dockerfile)

with open("requirements.txt", "w", encoding="utf-8") as file:
    file.write(requirements)

print("Dockerfile created successfully")
print("requirements.txt created successfully")

Dockerfile created successfully
requirements.txt created successfully
